# Dependencies

Install the required dependencies and libraries

In [60]:
!pip3 install langchain langchain_core langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 132.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Environment Variables and Constants

Initialize the constants and environment variables first

In [65]:
import os

huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

TEMPLATE = """
You are an AI security analyst.
You will be provided vulnerability scanner data in SARIF format with vulnerabilites detected in assets and services.
Alongside that, you will also be provided with the artifacts information such as assets information and service and architecture information and description.
Your job is to analyze the actual risk posed by the vulnerabilites by taking in the contextual information from the artifacts.

Scanner: {scanner}

Assets: {assets}

Exploit Data: {exploit}

SBOM: {sbom}

Architecture: {architecture}

Service Documentation: {service_docs}

Service Catalogue: {service_catalogue}
"""

# Requirements

Install the requirements and import relevant modules

In [81]:
import json
from pathlib import Path
from typing import List, Dict
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Data Loaders

Methods used for loading the data

In [27]:
def load_json_files() -> Dict:
    """
    Loads vulnerability and asset data from the json files

    Returns:
        Dictionary of the data files with their data content
    """

    # A dictionary to hold different datasets
    data_registry = {}
    
    # Path to data folder
    data_folder = Path("data")
    
    for file_path in data_folder.glob("*.json"):
        with open(file_path, 'r', encoding='utf-8') as f:

            data_registry[file_path.stem] = json.load(f)

    return data_registry

# Setup Workflow

These methods setup the basic model workflow

In [84]:
def create_prompt(template) -> ChatPromptTemplate:
    """
    Sets up the chat prompt to be used with the model

    Args:
        * template (str): The prompt template to use
        
    Returns:
        `ChatPromptTemplate` object
    """

    prompt = ChatPromptTemplate.from_template(template) 
    return prompt

def load_model_from_hf(repo_id: str) -> HuggingFaceEndpoint:
    """
    Connects to the model using the HuggingFace Inference API.

    Returns:
        A `HuggingFaceEndpoint` model
    """
    
    llm = HuggingFaceEndpoint(repo_id=repo_id, huggingfacehub_api_token=huggingfacehub_api_token)
    model = ChatHuggingFace(llm=llm)
    return model

In [85]:
model = load_model_from_hf("deepseek-ai/DeepSeek-R1-Distill-Qwen-32B")

In [91]:
model.invoke("tell me a joke")

AIMessage(content="\n\nWhy don't skeletons fight each other?  \nBecause they don't have the guts!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 210, 'prompt_tokens': 9, 'total_tokens': 219}, 'model_name': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d092a-fadb-73f0-b8bc-cb229e90546e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 210, 'total_tokens': 219})